# 02 · Whiplash index and April 1 snowpack

Full-record series per region, every winter 1950-51 onward, El Niño winters
highlighted (`ENSO_INDEX` from `src/config.py`). Descriptive only: no tests
before the freeze, and no fire data.

Inputs are built by code before this notebook runs (both cached in
`data/processed/`):

- `python -m src.whiplash`: Thornthwaite PET and SPEI-3 per ERA5-Land cell,
  log-logistic fit over 1951-2020, clipped to ±3.09, cos(latitude) regional
  means re-standardised over 1951-2020; whiplash = SPEI-3(JJA) - SPEI-3(DJF).
- `python -m src.snow`: April 1 00:00 UTC SWE from hourly ERA5-Land, glacier
  and perennial-snow cells excluded.
- `src.phase2`: the joined region x year table, written below to
  `data/processed/` and `results/tables/`.

Settings and their measured effects: `docs/methods_register.md`.

In [ ]:
import pandas as pd

from src.config import ENSO_INDEX, RESULTS
from src.describe import phase_summary
from src.figures import FIGURES, winter_series
from src.phase2 import SWE, WHIPLASH, build_table, write_tables

table = build_table(pd.read_csv(WHIPLASH), pd.read_csv(SWE))
write_tables(table)  # data/processed/ and results/tables/
print(f"{len(table)} rows, years {table['year'].min()}-{table['year'].max()}, index {ENSO_INDEX}")
table.head()

## Glacier exclusion

Mean April 1 SWE per region with glacier and perennial-snow cells excluded,
against all cells. The excluded cells add depth without year-to-year change,
so they inflate the mean while barely touching the variability.

In [ ]:
glacier = table.groupby("region", sort=False)[["swe_apr1_mm", "swe_apr1_mm_all_cells"]].agg(["mean", "std"])
glacier.round(1)

## Whiplash, every year

Year Y pairs winter (Y-1)-Y with summer Y. Negative = a swing from a wetter
than usual winter to a drier than usual summer. 2026 has a winter value but
no summer yet (ERA5-Land ends July 2026).

In [ ]:
winter_series(
    table.dropna(subset=["whiplash"]), "whiplash", "SPEI-3 JJA minus DJF (SD)", ENSO_INDEX,
    FIGURES / f"whiplash_series_{ENSO_INDEX.lower()}.png", signed=True,
    xlabel="Winter (Dec-Feb) and the following summer (Jun-Aug); year Y = winter (Y-1)-Y and summer Y")

## April 1 SWE, every winter

SoCal is left out of this figure: its regional April 1 SWE is close to zero
in every winter. It stays in the table.

In [ ]:
winter_series(
    table[table["region"] != "SoCal"].dropna(subset=["swe_apr1_mm"]), "swe_apr1_mm",
    "April 1 SWE (mm)", ENSO_INDEX, FIGURES / f"swe_april1_series_{ENSO_INDEX.lower()}.png")

## Descriptive phase summary

Group means and El Niño differences, in the variable's units, as % of the
all-winter mean and in SDs of all winters. No p-values: those come after the
freeze.

In [ ]:
cols = ["region", "n_el_nino", "n_strong", "n_neutral", "mean_all", "mean_el_nino", "mean_neutral",
        "nino_minus_neutral_pct", "nino_minus_neutral_sd", "strong_minus_all_pct",
        "strong_minus_all_sd", "nino_share_below_median"]
phase_summary(table[table["region"] != "SoCal"].dropna(subset=["swe_apr1_mm"]), "swe_apr1_mm")[cols].round(2)

In [ ]:
wcols = ["region", "n_el_nino", "n_strong", "mean_el_nino", "mean_neutral", "mean_strong",
         "nino_minus_neutral", "nino_minus_neutral_sd", "strong_minus_all_sd"]
w = table.dropna(subset=["whiplash"])
pd.concat({v: phase_summary(w, v)[wcols].set_index("region")
           for v in ["spei3_djf", "spei3_jja", "whiplash"]}, axis=1).round(2)